# MLP on TF-IDF with Hidden Layer Optimization

This notebook tunes hidden-layer configurations for an MLP classifier and evaluates the best model on a held-out test set.

## Why SVD before MLP?

TF-IDF is high-dimensional and sparse. MLP works better with dense, lower-dimensional inputs.
We apply `TruncatedSVD` to reduce dimensionality before MLP.

In [5]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.sparse import load_npz
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [6]:
from pathlib import Path
cwd = Path.cwd().resolve()

# Search upward so this notebook works from project root, notebooks/, or notebooks/experiments/.
PROJECT_ROOT = None
for candidate in [cwd, *cwd.parents]:
    if (candidate / "data").exists() and (candidate / "requirements.txt").exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"Could not resolve project root from {cwd}. "
        "Open the notebook from within the hate-speech-detection workspace."
    )

TFIDF_DIR = PROJECT_ROOT / "data" / "processed" / "tfidf"
X_PATH = TFIDF_DIR / "X_tfidf.npz"
Y_PATH = TFIDF_DIR / "y_labels.csv"
if not X_PATH.exists() or not Y_PATH.exists():
    raise FileNotFoundError(
        "Missing TF-IDF artifacts. Run these first:\n"
        "1) .\\.venv\\Scripts\\python.exe scripts/preprocess_data.py\n"
        "2) .\\.venv\\Scripts\\python.exe scripts/vectorize_tfidf.py"
    )

In [7]:
X = load_npz(X_PATH)
y = pd.read_csv(Y_PATH)["class"].to_numpy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print(f"X shape: {X.shape}")
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print("Class distribution:")
print(pd.Series(y).value_counts().sort_index())

X shape: (24783, 20000)
Train: (19826, 20000), Test: (4957, 20000)
Class distribution:
0     1430
1    19190
2     4163
Name: count, dtype: int64


In [8]:
pipeline = Pipeline(
    steps=[
        ("svd", TruncatedSVD(n_components=300, random_state=42)),
        ("scaler", StandardScaler()),
        (
            "mlp",
            MLPClassifier(
                max_iter=300,
                early_stopping=True,
                validation_fraction=0.1,
                n_iter_no_change=10,
                random_state=42,
            ),
        ),
    ]
)

param_grid = {
    "mlp__hidden_layer_sizes": [
        (64,),
        (128,),
        (128, 64),
        (256, 128),
    ],
    "mlp__alpha": [1e-4, 1e-3],
    "mlp__learning_rate_init": [1e-3],
}

grid = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring="f1_macro",
    cv=2,
    n_jobs=1,
    verbose=1,
)

grid.fit(X_train, y_train)
best_model = grid.best_estimator_

print("Best params:", grid.best_params_)
print(f"Best CV f1_macro: {grid.best_score_:.4f}")

Fitting 2 folds for each of 8 candidates, totalling 16 fits
Best params: {'mlp__alpha': 0.0001, 'mlp__hidden_layer_sizes': (64,), 'mlp__learning_rate_init': 0.001}
Best CV f1_macro: 0.6770


In [9]:
cv_results = pd.DataFrame(grid.cv_results_)
cols = [
    "rank_test_score",
    "mean_test_score",
    "std_test_score",
    "param_mlp__hidden_layer_sizes",
    "param_mlp__alpha",
    "param_mlp__learning_rate_init",
]
cv_results[cols].sort_values("rank_test_score").head(10)

,rank_test_score,mean_test_score,std_test_score,param_mlp__hidden_layer_sizes,param_mlp__alpha,param_mlp__learning_rate_init
0,1,0.676953,0.009515,"(64,)",0.0001,0.001
4,2,0.676895,0.001954,"(64,)",0.0010,0.001
1,3,0.676017,0.009083,"(128,)",0.0001,0.001
5,4,0.674235,0.007434,"(128,)",0.0010,0.001
2,5,0.672352,0.003997,"(128, 64)",0.0001,0.001
6,6,0.668954,0.000420,"(128, 64)",0.0010,0.001
3,7,0.660096,0.003055,"(256, 128)",0.0001,0.001
7,8,0.659625,0.004635,"(256, 128)",0.0010,0.001


In [10]:
y_pred = best_model.predict(X_test)

metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "precision_macro": precision_score(y_test, y_pred, average="macro", zero_division=0),
    "recall_macro": recall_score(y_test, y_pred, average="macro", zero_division=0),
    "f1_macro": f1_score(y_test, y_pred, average="macro", zero_division=0),
    "f1_weighted": f1_score(y_test, y_pred, average="weighted", zero_division=0),
}

print("Test metrics (best MLP):")
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")

print("\nClassification report:")
print(classification_report(y_test, y_pred, digits=4, zero_division=0))

Test metrics (best MLP):
accuracy: 0.8804
precision_macro: 0.7060
recall_macro: 0.6581
f1_macro: 0.6691
f1_weighted: 0.8713

Classification report:
              precision    recall  f1-score   support

           0     0.4203    0.2028    0.2736       286
           1     0.9196    0.9419    0.9306      3838
           2     0.7782    0.8295    0.8030       833

    accuracy                         0.8804      4957
   macro avg     0.7060    0.6581    0.6691      4957
weighted avg     0.8670    0.8804    0.8713      4957



In [11]:
labels = np.sort(np.unique(y))
cm = confusion_matrix(y_test, y_pred, labels=labels)
cm_df = pd.DataFrame(cm, index=[f"true_{l}" for l in labels], columns=[f"pred_{l}" for l in labels])
cm_df

,pred_0,pred_1,pred_2
true_0,58,182,46
true_1,72,3615,151
true_2,8,134,691


## Optional next step

Compare this notebook's test metrics to your logistic regression and SVM notebooks to choose the best baseline for macro-F1 and class-0 performance.